# 🎵 Analisis Sentimen Ulasan Spotify — BiLSTM (Deep Learning)

**Mata Kuliah:** SD25-32202 Pemrosesan Bahasa Alami  
**Institusi:** Institut Teknologi Sumatera (ITERA)  
**Framework:** PyTorch  
**Arsitektur:** BiLSTM 2-layer (~2M parameters, ≤ 10M ✅)

## Tujuan Notebook
Melatih model Deep Learning (BiLSTM) untuk klasifikasi 3 kelas sentimen ulasan Spotify, sebagai pembanding terhadap model ML (Decision Tree + TF-IDF) yang sudah dilatih sebelumnya.

## Limitasi (akan dijelaskan di paper)
Model DL dilatih pada **subset 20K sample** dari dataset asli (~100K) karena keterbatasan compute lokal. Model ML dilatih pada full dataset 100K. Perbandingan dilakukan dengan kesadaran limitasi ini.

## 1. Setup & Import Library

In [ ]:
# Install dependencies (uncomment kalau belum terinstall)
# !pip install torch pandas scikit-learn matplotlib seaborn PySastrawi tqdm

In [1]:
import os
import re
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

PyTorch version: 2.2.2+cpu
Device: cpu


## 2. Load Dataset & Sampling 20K

Pakai **stratified sampling** supaya proporsi kelas tetap terjaga.

In [2]:
# Load dataset asli
DATA_PATH = '../../data/ulasan_com.spotify.music.csv'
df = pd.read_csv(DATA_PATH)
print(f'Total data: {len(df):,}')
df.head()

Total data: 100,000


,Nama User,Ulasan,Rating,Tanggal,Likes,Versi App
0,Pengguna Google,suka banget sama ni app,5,2025-12-31 17:27:52,0,9.1.6.1145
1,Pengguna Google,bgus,5,2025-12-31 17:25:25,0,9.1.6.1145
2,Pengguna Google,ga bisa denger lagu JKT48 lagi gara² selalu di...,1,2025-12-31 16:39:10,0,9.1.6.1145
3,Pengguna Google,lagunya bagus banget buat ngapa ngapain,5,2025-12-31 16:35:54,0,9.1.6.1145
4,Pengguna Google,"kenapa sih, aku cuman bisa main 5 lagu doang? ...",1,2025-12-31 16:29:15,0,9.1.6.1145


In [4]:
print('Kolom dataset:', df.columns.tolist())
print('\nPreview:')
df.head()

Kolom dataset: ['Nama User', 'Ulasan', 'Rating', 'Tanggal', 'Likes', 'Versi App']

Preview:


,Nama User,Ulasan,Rating,Tanggal,Likes,Versi App
0,Pengguna Google,suka banget sama ni app,5,2025-12-31 17:27:52,0,9.1.6.1145
1,Pengguna Google,bgus,5,2025-12-31 17:25:25,0,9.1.6.1145
2,Pengguna Google,ga bisa denger lagu JKT48 lagi gara² selalu di...,1,2025-12-31 16:39:10,0,9.1.6.1145
3,Pengguna Google,lagunya bagus banget buat ngapa ngapain,5,2025-12-31 16:35:54,0,9.1.6.1145
4,Pengguna Google,"kenapa sih, aku cuman bisa main 5 lagu doang? ...",1,2025-12-31 16:29:15,0,9.1.6.1145


In [ ]:
# Mapping rating -> label sentimen (sama dengan ML)
# 1,2 -> Negatif | 3 -> Netral | 4,5 -> Positif
SENTIMENT_MAPPING = {1: 'Negatif', 2: 'Negatif', 3: 'Netral', 4: 'Positif', 5: 'Positif'}
LABEL_TO_IDX = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

# Samakan nama kolom ulasan agar konsisten dengan pipeline berikutnya
if 'content' not in df.columns and 'Ulasan' in df.columns:
    df = df.rename(columns={'Ulasan': 'content'})

df['sentiment'] = df['Rating'].map(SENTIMENT_MAPPING)
df['label'] = df['sentiment'].map(LABEL_TO_IDX)

# Drop NaN dan duplicates
df = df.dropna(subset=['content', 'label']).drop_duplicates(subset='content')
df = df[['content', 'label', 'sentiment']].reset_index(drop=True)

print(f'Setelah cleaning: {len(df):,}')
print('\nDistribusi label:')
print(df['sentiment'].value_counts())

KeyError: ['content']

In [ ]:
# Stratified sampling 20K
SAMPLE_SIZE = 20000

df_sampled, _ = train_test_split(
    df,
    train_size=SAMPLE_SIZE,
    stratify=df['label'],
    random_state=SEED
)
df_sampled = df_sampled.reset_index(drop=True)

print(f'Sample size: {len(df_sampled):,}')
print('\nDistribusi label (sampled):')
print(df_sampled['sentiment'].value_counts())
print('\nProporsi (%):')
print((df_sampled['sentiment'].value_counts(normalize=True) * 100).round(2))

## 3. Preprocessing (Sama dengan ML — fair comparison)

In [ ]:
# Load preprocessing info dari ML (slang_dict + stopwords)
with open('../../ml/deployment/preprocessing_info.pkl', 'rb') as f:
    preprocess_info = pickle.load(f)

slang_dict = preprocess_info['slang_dict']
stopwords_id = set(preprocess_info['stopwords'])

print(f'Slang dict: {len(slang_dict)} entries')
print(f'Stopwords: {len(stopwords_id)} entries')

# Stemmer
stemmer = StemmerFactory().create_stemmer()

In [ ]:
def clean_text(text):
    """Pipeline preprocessing — sama persis dengan deployment ML."""
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [slang_dict.get(w, w) for w in words]
    words = [w for w in words if w not in stopwords_id and len(w) > 1]
    words = [stemmer.stem(w) for w in words]
    text = ' '.join(words)
    return re.sub(r'\s+', ' ', text).strip()

# Apply preprocessing (memakan waktu ~5-10 menit untuk 20K karena Sastrawi)
tqdm.pandas(desc='Preprocessing')
df_sampled['cleaned'] = df_sampled['content'].progress_apply(clean_text)

# Drop empty after cleaning
df_sampled = df_sampled[df_sampled['cleaned'].str.len() > 0].reset_index(drop=True)
print(f'\nFinal size after cleaning: {len(df_sampled):,}')
df_sampled[['content', 'cleaned', 'sentiment']].head()

## 4. Build Vocabulary & Tokenization

In [ ]:
# Build vocabulary dari training data nanti, tapi dulu kita split dulu
X = df_sampled['cleaned'].values
y = df_sampled['label'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, stratify=y_temp, random_state=SEED
)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

In [ ]:
# Build vocab dari training set saja (hindari data leakage)
MIN_FREQ = 2          # ignore kata yang muncul < 2x
MAX_VOCAB = 10000     # ambil top-N kata

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

counter = Counter()
for text in X_train:
    counter.update(text.split())

# Filter by frequency
vocab_words = [w for w, c in counter.most_common(MAX_VOCAB - 2) if c >= MIN_FREQ]

# word2idx (PAD=0, UNK=1)
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for w in vocab_words:
    word2idx[w] = len(word2idx)

VOCAB_SIZE = len(word2idx)
print(f'Vocab size: {VOCAB_SIZE:,}')
print(f'Sample top-10: {vocab_words[:10]}')

In [ ]:
# Analisis panjang sequence
lengths = [len(t.split()) for t in X_train]
print(f'Mean: {np.mean(lengths):.1f} | Median: {np.median(lengths):.0f} | 95th %ile: {np.percentile(lengths, 95):.0f} | Max: {max(lengths)}')

# Set MAX_LEN sesuai 95th percentile (covering most data)
MAX_LEN = int(np.percentile(lengths, 95))
print(f'\nMAX_LEN dipilih: {MAX_LEN}')

In [ ]:
def text_to_indices(text, word2idx, max_len):
    """Convert text → list of indices, truncate/pad ke max_len."""
    tokens = text.split()[:max_len]
    indices = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    # Pad
    if len(indices) < max_len:
        indices += [word2idx[PAD_TOKEN]] * (max_len - len(indices))
    return indices

# Test
sample_text = X_train[0]
sample_idx = text_to_indices(sample_text, word2idx, MAX_LEN)
print(f'Text: {sample_text}')
print(f'Indices (first 20): {sample_idx[:20]}')
print(f'Length: {len(sample_idx)}')

## 5. PyTorch Dataset & DataLoader

In [ ]:
class SpotifyDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        indices = text_to_indices(self.texts[idx], self.word2idx, self.max_len)
        return (
            torch.tensor(indices, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

BATCH_SIZE = 64

train_ds = SpotifyDataset(X_train, y_train, word2idx, MAX_LEN)
val_ds   = SpotifyDataset(X_val, y_val, word2idx, MAX_LEN)
test_ds  = SpotifyDataset(X_test, y_test, word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check
x_batch, y_batch = next(iter(train_loader))
print(f'Batch X: {x_batch.shape} | Batch y: {y_batch.shape}')
print(f'Sample y: {y_batch[:8].tolist()}')

## 6. Arsitektur BiLSTM

```
Embedding(vocab=10K, dim=128)         → ~1.28M params
       ↓
BiLSTM(input=128, hidden=128, layers=2) → ~0.66M params
       ↓
Dropout(0.5)
       ↓
Linear(256→64) + ReLU                  → ~16K params
       ↓
Dropout(0.3)
       ↓
Linear(64→3)                           → ~195 params
       ↓
Softmax → [Negatif, Netral, Positif]

Total: ~2M params (jauh di bawah 10M ✅)
```

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, num_classes=3, dropout=0.5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)  # *2 karena bidirectional
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)              # (batch, seq_len, embed_dim)
        lstm_out, (h_n, c_n) = self.lstm(emb) # lstm_out: (batch, seq_len, hidden*2)
        
        # Ambil last hidden state (concat forward & backward)
        # h_n shape: (num_layers*2, batch, hidden)
        # Kita ambil 2 layer terakhir (forward + backward dari layer terakhir)
        h_forward = h_n[-2, :, :]   # (batch, hidden)
        h_backward = h_n[-1, :, :]  # (batch, hidden)
        h_concat = torch.cat([h_forward, h_backward], dim=1)  # (batch, hidden*2)
        
        out = self.dropout1(h_concat)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        logits = self.fc2(out)  # (batch, num_classes)
        return logits

# Init model
model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    hidden_dim=128,
    num_layers=2,
    num_classes=3,
    dropout=0.5
).to(DEVICE)

# Hitung total parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)')
print(f'\n✅ Constraint ≤ 10M: {"PASS" if total_params <= 10_000_000 else "FAIL"}')
print(model)

## 7. Training Loop

Strategi:
- **Optimizer:** Adam, lr=1e-3
- **Loss:** CrossEntropyLoss
- **Early stopping:** patience=3 (stop kalau val_loss tidak membaik)
- **Max epoch:** 10

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in tqdm(loader, desc='Train', leave=False):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds, all_labels = [], []
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    
    return total_loss / total, correct / total, all_preds, all_labels

In [ ]:
# Hyperparameters
EPOCHS = 10
LR = 1e-3
PATIENCE = 3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print(f'Training on {DEVICE}\n')
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  → ✅ Best model saved (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        print(f'  → ⏳ Patience: {patience_counter}/{PATIENCE}')
        if patience_counter >= PATIENCE:
            print(f'\n🛑 Early stopping triggered at epoch {epoch}')
            break

# Restore best model
model.load_state_dict(best_model_state)
model = model.to(DEVICE)
print('\n✅ Best model restored.')

## 8. Visualisasi Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_run = list(range(1, len(history['train_loss']) + 1))

axes[0].plot(epochs_run, history['train_loss'], 'o-', label='Train Loss')
axes[0].plot(epochs_run, history['val_loss'], 's-', label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Training & Validation Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_run, history['train_acc'], 'o-', label='Train Acc')
axes[1].plot(epochs_run, history['val_acc'], 's-', label='Val Acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Training & Validation Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluasi pada Test Set

In [ ]:
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='weighted'
)

print('=' * 50)
print('📊 EVALUASI MODEL BiLSTM PADA TEST SET')
print('=' * 50)
print(f'Test Loss      : {test_loss:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')
print(f'Test Precision : {precision:.4f}')
print(f'Test Recall    : {recall:.4f}')
print(f'Test F1-Score  : {f1:.4f}')
print('=' * 50)

print('\nClassification Report:')
target_names = [IDX_TO_LABEL[i] for i in range(3)]
print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('BiLSTM — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig('confusion_matrix_dl.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Model & Vocab untuk Deployment

In [ ]:
# Save model state_dict (lebih portable daripada full model)
OUTPUT_DIR = '../deployment'
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_path = f'{OUTPUT_DIR}/model_bilstm.pt'
torch.save({
    'state_dict': model.state_dict(),
    'config': {
        'vocab_size': VOCAB_SIZE,
        'embed_dim': 128,
        'hidden_dim': 128,
        'num_layers': 2,
        'num_classes': 3,
        'dropout': 0.5,
        'max_len': MAX_LEN,
    },
}, model_path)
print(f'✅ Model saved: {model_path}')

# Save vocab
vocab_path = f'{OUTPUT_DIR}/vocab.pkl'
with open(vocab_path, 'wb') as f:
    pickle.dump({
        'word2idx': word2idx,
        'idx_to_label': IDX_TO_LABEL,
        'max_len': MAX_LEN,
    }, f)
print(f'✅ Vocab saved: {vocab_path}')

# Save preprocessing info (copy dari ML)
import shutil
shutil.copy('../../ml/deployment/preprocessing_info.pkl', f'{OUTPUT_DIR}/preprocessing_info.pkl')
print(f'✅ Preprocessing info copied: {OUTPUT_DIR}/preprocessing_info.pkl')

# Save training metrics untuk paper
metrics = {
    'model_name': 'BiLSTM',
    'total_params': total_params,
    'sample_size': len(df_sampled),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'epochs_trained': len(history['train_loss']),
    'final_train_loss': history['train_loss'][-1],
    'final_val_loss': history['val_loss'][-1],
    'test_accuracy': test_acc,
    'test_precision': precision,
    'test_recall': recall,
    'test_f1': f1,
    'history': history,
}
metrics_df = pd.DataFrame([{
    'Model': 'BiLSTM',
    'Accuracy': round(test_acc, 4),
    'Precision': round(precision, 4),
    'Recall': round(recall, 4),
    'F1-Score': round(f1, 4),
    'Params': f'{total_params:,}'
}])
metrics_df.to_csv('hasil_benchmark_dl.csv', index=False)
print(f'\n✅ Hasil benchmark disimpan: hasil_benchmark_dl.csv')
print('\n📊 Ringkasan:')
print(metrics_df.to_string(index=False))

## 11. Inference Test (Sanity Check sebelum Deploy)

In [ ]:
@torch.no_grad()
def predict_text(text, model, word2idx, max_len, device):
    model.eval()
    cleaned = clean_text(text)
    if not cleaned:
        return None, None, cleaned
    indices = text_to_indices(cleaned, word2idx, max_len)
    x = torch.tensor([indices], dtype=torch.long).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    return IDX_TO_LABEL[pred_idx], float(probs[pred_idx]), cleaned

tests = [
    'suka banget sama spotify, lagunya lengkap dan enak didengar',
    'banyak iklan sangat mengganggu, tidak bisa mendengarkan lagu',
    'lumayan sih tapi masih perlu perbaikan',
    'error terus gabisa dibuka, tolong perbaiki',
    'biasa aja, masih banyak bug',
]

for t in tests:
    label, conf, cleaned = predict_text(t, model, word2idx, MAX_LEN, DEVICE)
    print(f'\nInput   : {t}')
    print(f'Cleaned : {cleaned}')
    print(f'Predict : {label} (conf: {conf:.2%})')

---

## ✅ Notebook selesai

**Yang sudah dihasilkan:**
- `../deployment-dl/model_bilstm.pt` — checkpoint model PyTorch
- `../deployment-dl/vocab.pkl` — word2idx, idx_to_label, max_len
- `../deployment-dl/preprocessing_info.pkl` — slang_dict, stopwords
- `hasil_benchmark_dl.csv` — metrik evaluasi untuk paper
- `training_curves.png`, `confusion_matrix_dl.png` — figure untuk paper

**Langkah selanjutnya:**
1. Test `app.py` di lokal: `streamlit run deployment-dl/app.py`
2. Deploy ke Hugging Face Space baru: `sentimen-spotify-dl`
3. Update README.md repo GitHub dengan link Space DL